# Exploration Notebook
## Letterboxd Dashboard
### Reid B.

In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
ratings   = pd.read_csv('data/ratings.csv')
diary     = pd.read_csv('data/diary.csv')
watched   = pd.read_csv('data/watched.csv')
watchlist = pd.read_csv('data/watchlist.csv')

for df in [ratings, diary, watched, watchlist]:
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    if 'year' in df.columns:
        df['year'] = df['year'].astype('Int64')

diary['watched_date'] = pd.to_datetime(diary['watched_date'], errors='coerce')
diary['date']         = pd.to_datetime(diary['date'], errors='coerce')
diary_dated           = diary.dropna(subset=['watched_date'])
ratings['decade']     = (ratings['year'] // 10 * 10).astype(str) + 's'

print(f'Ratings: {ratings.shape}')
print(f'Diary:   {diary.shape}')

## 1. Rating Distribution
How do I rate films — am I generous or harsh?

In [ ]:
rc = ratings['rating'].value_counts().sort_index().reset_index()
rc.columns = ['rating', 'count']

fig = px.bar(
    rc,
    x='rating',
    y='count',
    title='My rating distribution',
    labels={'rating': 'Rating (★)', 'count': 'Films'},
    color='count',
    color_continuous_scale='Oranges',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 2. Most Watched Release Years
Which years of film do I watch most?

In [ ]:
year_counts = ratings['year'].value_counts().reset_index()
year_counts.columns = ['year', 'count']
year_counts = year_counts.sort_values('year')

fig = px.bar(
    year_counts,
    x='year',
    y='count',
    title='Films watched by release year',
    labels={'year': 'Release year', 'count': 'Films'},
    color='count',
    color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 3. Do I Rate Older Films Higher?
Average rating by release year, filtered to years with 3+ films.

In [ ]:
year_ratings = (
    ratings.groupby('year')['rating']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_rating', 'count': 'films'})
    .query('films >= 3')
    .sort_values('year')
)
year_ratings['avg_rating'] = year_ratings['avg_rating'].round(2)

fig = px.scatter(
    year_ratings,
    x='year',
    y='avg_rating',
    size='films',
    title='Avg rating by release year (min 3 films)',
    labels={'year': 'Release year', 'avg_rating': 'Avg rating', 'films': 'Films watched'},
    color='avg_rating',
    color_continuous_scale='RdYlGn',
    hover_data=['films']
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 4. Average Rating by Decade
Do I prefer films from certain eras?

In [ ]:
decade_stats = (
    ratings.groupby('decade')['rating']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_rating', 'count': 'films'})
    .query('films >= 5')
    .sort_values('decade')
)
decade_stats['avg_rating'] = decade_stats['avg_rating'].round(2)

fig = px.bar(
    decade_stats,
    x='decade',
    y='avg_rating',
    title='Avg rating by decade (min 5 films)',
    labels={'decade': 'Decade', 'avg_rating': 'Avg rating', 'films': 'Films'},
    color='avg_rating',
    color_continuous_scale='Purples',
    text='avg_rating',
    hover_data=['films']
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, yaxis_range=[0, 5])
fig.show()

## 5. Films Logged Per Month
How has my watching habit changed over time?

In [ ]:
monthly = (
    diary_dated
    .groupby(diary_dated['watched_date'].dt.to_period('M'))
    .size()
    .reset_index(name='films')
)
monthly['watched_date'] = monthly['watched_date'].astype(str)

fig = px.line(
    monthly,
    x='watched_date',
    y='films',
    title='Films watched per month',
    labels={'watched_date': 'Month', 'films': 'Films watched'},
    markers=True
)
fig.update_xaxes(tickangle=45)
fig.update_traces(line_color='#e63946')
fig.show()

## 6. Films by Decade (Count)
Which decade of cinema do I watch most?

In [ ]:
decade_counts = (
    ratings['decade']
    .value_counts()
    .sort_index()
    .reset_index()
)
decade_counts.columns = ['decade', 'count']

fig = px.bar(
    decade_counts,
    x='decade',
    y='count',
    title='Films watched by decade',
    labels={'decade': 'Decade', 'count': 'Films'},
    color='count',
    color_continuous_scale='Teal',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 7. Most Rewatched Films
Which films have I come back to most?

In [ ]:
rewatched = (
    diary.groupby(['name', 'year'])
    .size()
    .reset_index(name='watches')
    .query('watches > 1')
    .sort_values('watches', ascending=False)
    .head(15)
)
rewatched['label'] = rewatched['name'] + ' (' + rewatched['year'].astype(str) + ')'

fig = px.bar(
    rewatched,
    x='watches',
    y='label',
    orientation='h',
    title='Most rewatched films',
    labels={'watches': 'Times watched', 'label': ''},
    color='watches',
    color_continuous_scale='Reds',
    text='watches'
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False
)
fig.show()

## 8. Films Watched Per Year (Diary)
How many films did I actually watch each calendar year?

In [ ]:
per_year = (
    diary_dated
    .groupby(diary_dated['watched_date'].dt.year)
    .size()
    .reset_index(name='films')
    .rename(columns={'watched_date': 'year'})
)

fig = px.bar(
    per_year,
    x='year',
    y='films',
    title='Films watched per calendar year',
    labels={'year': 'Year', 'films': 'Films watched'},
    color='films',
    color_continuous_scale='Greens',
    text='films'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, xaxis=dict(tickmode='linear', dtick=1))
fig.show()